## 1. Định Nghĩa Hệ Thống & Bài Toán (System and Problem Definition)

**Hệ thống thông minh:** Hệ thống hỗ trợ chẩn đoán sớm nguy cơ mắc bệnh đái tháo đường (Type 2 Diabetes) dựa trên các chỉ số sinh hóa lâm sàng.

**Phát biểu bài toán hình thức:** Cho trước vector đặc trưng $\mathbf{x} \in \mathbb{R}^8$ biểu diễn hồ sơ bệnh nhân, tìm hàm $f_\theta(\mathbf{x}): \mathbb{R}^8 \rightarrow \{0, 1\}$ để dự đoán chính xác nhãn mục tiêu $y$ (0: Không mắc bệnh, 1: Mắc bệnh tiểu đường) cho quan sát mới chưa từng thấy.

## 2. Sơ Đồ Hệ Thống Thông Minh (Intelligent System Diagram)

```
[ Môi trường: Bệnh nhân ] ---> [ Nhận thức: Đo đạc 8 chỉ số ] ---> [ Biểu diễn: Vector x in R^8 ]
                                                                          |
                                                                          v
[ Ứng dụng Web/Mobile ] <--- [ Quyết định: Cảnh báo & Xác suất ] <--- [ Mô hình: Random Forest ]
                                            ^
                                            |
                         [ Đồ thị Tri thức Neo4j Long Châu ]
```

In [ ]:
# Hiển thị sơ đồ kiến trúc hệ thống
print("Kiến trúc luồng xử lý: Real-world Data -> Feature Vector -> Traditional ML -> Inference -> Decision")


## 3. Nguồn Dữ Liệu Thực Tế (Dataset Source)

• **Tên tập dữ liệu:** Pima Indians Diabetes Database
• **Cơ quan phát hành:** National Institute of Diabetes and Digestive and Kidney Diseases (NIDDK), lưu trữ tại Kaggle / UCI ML Repository.
• **URL:** https://www.kaggle.com/datasets/uciml/pima-indians-diabetes-database
• **Quy mô:** 768 bệnh nhân nữ người Pima từ 21 tuổi trở lên.

In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# Nạp dữ liệu với cơ chế fallback linh hoạt
data_path = os.path.join('data', 'diabetes.csv')
if not os.path.exists(data_path):
    data_path = os.path.join('..', '..', 'DATA', 'diabetes.csv')
df = pd.read_csv(data_path)
print("Kích thước dữ liệu gốc:", df.shape)
df.head()


## 4. Mô Tả Tập Dữ Liệu (Dataset Description)

**Giải đáp 10 câu hỏi dữ liệu bắt buộc:**
1. *Hiện tượng thực tế:* Tình trạng chuyển hóa đường và nguy cơ đái tháo đường ở phụ nữ Pima.
2. *Một quan sát:* Một bản ghi hồ sơ khám sức khỏe của một bệnh nhân.
3. *Các đặc trưng:* Pregnancies, Glucose, BloodPressure, SkinThickness, Insulin, BMI, DiabetesPedigreeFunction, Age.
4. *Biến mục tiêu:* `Outcome` (0: Âm tính, 1: Dương tính với tiểu đường).
5. *Kiểu biến mục tiêu:* Biến rời rạc nhị phân (Binary Categorical).
6. *Loại bài toán:* Học có giám sát - Phân loại nhị phân (Binary Classification).
7. *Số quan sát:* 768 bệnh nhân.
8. *Số đặc trưng:* 8 đặc trưng đầu vào.
9. *Đặc trưng số học:* Glucose, BloodPressure, SkinThickness, Insulin, BMI, DiabetesPedigreeFunction, Age.
10. *Đặc trưng rời rạc:* Pregnancies (số lần mang thai).

In [ ]:
print("--- THÔNG TIN TỔNG QUAN TẬP DỮ LIỆU ---")
print(df.info())
print("--- THỐNG KÊ MÔ TẢ ---")
df.describe().T


## 5. Biểu Diễn Dữ Liệu (Data Representation - The Central Idea)

Mỗi bệnh nhân được biểu diễn bởi một vector $d$-chiều $\mathbf{x}_i = [x_{i1}, x_{i2}, \dots, x_{i8}]^T \in \mathbb{R}^8$.

**Nguyên lý phân biệt 3 cấp độ:**
$$\text{Raw Feature (Đặc trưng thô)} \neq \text{Encoded Feature (Đặc trưng tiền xử lý)} \neq \text{Model Input (Đầu vào sau StandardScaler)}$$

Các giá trị 0 phi lý trong các cột sinh học (Glucose, BloodPressure, SkinThickness, Insulin, BMI) được phát hiện và thay thế bằng trung vị (Median Imputation) học từ tập Train.

In [ ]:
# Kiểm tra các giá trị 0 phi lý sinh học
zero_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
zero_counts = {col: (df[col] == 0).sum() for col in zero_cols}
print("Số lượng giá trị 0 phi lý trong từng đặc trưng sinh học:")
for col, cnt in zero_counts.items():
    print(f" - {col}: {cnt} mẫu ({cnt/len(df)*100:.1f}%)")


## 6. Phân Tích Đặc Trưng & Mục Tiêu (Feature and Target Analysis)

Phân tích tương quan Pearson giữa từng đặc trưng với biến mục tiêu `Outcome` để xác định các yếu tố có lực phân tách mạnh nhất đối với bệnh lý.

In [ ]:
corr = df.corr(numeric_only=True)
print("Tương quan với biến mục tiêu Outcome:")
print(corr['Outcome'].sort_values(ascending=False))


## 7. Phân Tích Dữ Liệu Khám Phá (Exploratory Data Analysis - EDA)

Trực quan hóa phân phối biến mục tiêu, phân phối các đặc trưng sinh học chính và ma trận tương quan nhiệt (Correlation Heatmap).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
palette = {0: '#3498db', 1: '#e74c3c'}

# Biểu đồ phân phối nhãn
sns.countplot(data=df, x='Outcome', hue='Outcome', palette=palette, legend=False, ax=axes[0])
axes[0].set_title('1. Phân Phối Nhãn (0: Khỏe mạnh, 1: Tiểu đường)', fontsize=12, fontweight='bold')
axes[0].set_xticks([0, 1])
axes[0].set_xticklabels(['0 - Không mắc bệnh', '1 - Mắc tiểu đường'])

# Phân phối Glucose theo Outcome
sns.kdeplot(data=df, x='Glucose', hue='Outcome', palette=palette, fill=True, common_norm=False, ax=axes[1])
axes[1].set_title('2. Phân Phối Glucose Theo Nhóm Bệnh Nhân', fontsize=12, fontweight='bold')

# Ma trận tương quan Heatmap
sns.heatmap(df.corr(numeric_only=True), annot=True, fmt='.2f', cmap='Blues', ax=axes[2], cbar=False)
axes[2].set_title('3. Ma Trận Tương Quan Pearson', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()


## 8. Phân Chia Tập Train / Test (Train/Test Split)

**Nguyên tắc khoa học:** Chia tỷ lệ 80% Training và 20% Testing với `stratify=y` và `random_state=42`.
**Ngăn ngừa rò rỉ dữ liệu (No Data Leakage):** Median Imputation và StandardScaler chỉ học trên tập Train (`fit_transform`) và chỉ áp dụng lại lên tập Test (`transform`).

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df.drop(columns=['Outcome'])
y = df['Outcome']

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Xử lý khuyết thiếu (Median Imputation) dựa trên tập Train
impute_values = {}
X_train_clean = X_train_raw.copy()
X_test_clean = X_test_raw.copy()

for col in zero_cols:
    median_val = X_train_raw[X_train_raw[col] > 0][col].median()
    impute_values[col] = median_val
    X_train_clean[col] = X_train_clean[col].replace(0, median_val)
    X_test_clean[col] = X_test_clean[col].replace(0, median_val)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_clean)
X_test = scaler.transform(X_test_clean)

print(f"Kích thước tập Train: {X_train.shape}, Tập Test: {X_test.shape}")
print("Tỷ lệ nhãn Train:", np.bincount(y_train) / len(y_train))
print("Tỷ lệ nhãn Test :", np.bincount(y_test) / len(y_test))


## 9. Đường Cơ Sở Tham Chiếu (Baseline Reference Point)

**Tại sao cần Baseline?** Mục đích là xác định xem mô hình học máy phức tạp có thực sự đem lại giá trị tri thức vượt trội so với một chiến lược phán đoán tầm thường (predict majority class) hay không.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
y_pred_base = baseline.predict(X_test)

print("--- HIỆU NĂNG MÔ HÌNH ĐỐI CHỨNG (BASELINE) ---")
print(f"Accuracy : {accuracy_score(y_test, y_pred_base):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_base, zero_division=0):.4f}")
print(f"Recall   : {recall_score(y_test, y_pred_base, zero_division=0):.4f}")
print(f"F1-Score : {f1_score(y_test, y_pred_base, zero_division=0):.4f}")
print(f"ROC-AUC  : {roc_auc_score(y_test, y_pred_base):.4f}")


## 10. Mô Hình 1: Logistic Regression

Học ranh giới phân tách tuyến tính qua hàm Sigmoid $\sigma(z) = \frac{1}{1 + e^{-(\mathbf{w}^T \mathbf{x} + b)}}$.

In [ ]:
from sklearn.linear_model import LogisticRegression

m1 = LogisticRegression(random_state=42, max_iter=1000)
m1.fit(X_train, y_train)
y_pred_m1 = m1.predict(X_test)
y_prob_m1 = m1.predict_proba(X_test)[:, 1]

print(f"Logistic Regression -> Acc: {accuracy_score(y_test, y_pred_m1):.4f}, F1: {f1_score(y_test, y_pred_m1):.4f}, AUC: {roc_auc_score(y_test, y_prob_m1):.4f}")


## 11. Mô Hình 2: k-Nearest Neighbors (k-NN)

Dự đoán nhãn dựa trên khoảng cách Euclidean tới $k$ láng giềng gần nhất trong không gian đặc trưng đã chuẩn hóa.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

m2 = KNeighborsClassifier(n_neighbors=7)
m2.fit(X_train, y_train)
y_pred_m2 = m2.predict(X_test)
y_prob_m2 = m2.predict_proba(X_test)[:, 1]

print(f"k-Nearest Neighbors (k=7) -> Acc: {accuracy_score(y_test, y_pred_m2):.4f}, F1: {f1_score(y_test, y_pred_m2):.4f}, AUC: {roc_auc_score(y_test, y_prob_m2):.4f}")


## 12. Mô Hình 3: Support Vector Machine (SVM)

Tìm siêu phẳng tối ưu phân tách cực đại hóa lề (Margin Maximization) với Kernel RBF.

In [ ]:
from sklearn.svm import SVC

m3 = SVC(kernel='rbf', probability=True, random_state=42)
m3.fit(X_train, y_train)
y_pred_m3 = m3.predict(X_test)
y_prob_m3 = m3.predict_proba(X_test)[:, 1]

print(f"Support Vector Machine (RBF) -> Acc: {accuracy_score(y_test, y_pred_m3):.4f}, F1: {f1_score(y_test, y_pred_m3):.4f}, AUC: {roc_auc_score(y_test, y_prob_m3):.4f}")


## 13. Mô Hình 4: Random Forest Classifier

Thuật toán Ensemble kết hợp Bagging và Random Subspace Feature Selection huấn luyện hàng trăm cây quyết định độc lập.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

m4 = RandomForestClassifier(n_estimators=150, max_depth=6, random_state=42)
m4.fit(X_train, y_train)
y_pred_m4 = m4.predict(X_test)
y_prob_m4 = m4.predict_proba(X_test)[:, 1]

print(f"Random Forest (Ensemble) -> Acc: {accuracy_score(y_test, y_pred_m4):.4f}, F1: {f1_score(y_test, y_pred_m4):.4f}, AUC: {roc_auc_score(y_test, y_prob_m4):.4f}")


## 14. Đánh Giá Toàn Diện (Comprehensive Evaluation)

Bảng so sánh 5 độ đo độc lập: Accuracy, Precision, Recall, F1-Score, ROC-AUC trên tập kiểm thử.

In [ ]:
models_dict = {
    'Baseline (Dummy)': (baseline, y_pred_base, y_pred_base),
    'Logistic Regression': (m1, y_pred_m1, y_prob_m1),
    'k-Nearest Neighbors': (m2, y_pred_m2, y_prob_m2),
    'Support Vector Machine': (m3, y_pred_m3, y_prob_m3),
    'Random Forest': (m4, y_pred_m4, y_prob_m4)
}

eval_results = []
for name, (model, y_pred, y_prob) in models_dict.items():
    eval_results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall': recall_score(y_test, y_pred, zero_division=0),
        'F1-Score': f1_score(y_test, y_pred, zero_division=0),
        'ROC-AUC': roc_auc_score(y_test, y_prob)
    })

eval_df = pd.DataFrame(eval_results)
eval_df


## 15. Thí Nghiệm 1: So Sánh Mô Hình (Controlled Experiment 1: Model Comparison)

**Câu hỏi thực nghiệm:** Mô hình nào có năng lực khái quát hóa tốt nhất và cân bằng giữa Precision/Recall trên dữ liệu y tế?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Biểu đồ so sánh F1 và Accuracy
eval_df.plot(x='Model', y=['Accuracy', 'F1-Score'], kind='bar', ax=axes[0], colormap='viridis')
axes[0].set_title('So Sánh Accuracy vs F1-Score Giữa Các Mô Hình', fontweight='bold')
axes[0].set_ylim(0, 1.0)
axes[0].set_ylabel('Điểm số')
axes[0].tick_params(axis='x', rotation=30)

# Confusion matrix của Random Forest
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, y_pred_m4)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1],
            xticklabels=['Dự đoán 0 (Âm tính)', 'Dự đoán 1 (Dương tính)'],
            yticklabels=['Thực tế 0', 'Thực tế 1'])
axes[1].set_title('Ma Trận Nhầm Lẫn (Random Forest)', fontweight='bold')

plt.tight_layout()
plt.show()


## 16. Thí Nghiệm 2: Khảo Sát Siêu Tham Số (Experiment 2: Hyperparameter Investigation)

**Câu hỏi thực nghiệm:** Độ sâu cây `max_depth` tác động như thế nào đến sự đánh đổi giữa Underfitting và Overfitting?

In [ ]:
depths = list(range(1, 15))
train_scores = []
test_scores = []

for d in depths:
    clf = RandomForestClassifier(max_depth=d, n_estimators=100, random_state=42)
    clf.fit(X_train, y_train)
    train_scores.append(clf.score(X_train, y_train))
    test_scores.append(clf.score(X_test, y_test))

plt.figure(figsize=(8, 4.5))
plt.plot(depths, train_scores, 'o-', label='Train Accuracy (Huấn luyện)', color='#e74c3c')
plt.plot(depths, test_scores, 's-', label='Test Accuracy (Kiểm thử)', color='#2980b9')
plt.axvline(x=6, color='gray', linestyle='--', label='Độ sâu tối ưu (max_depth=6)')
plt.title('Khảo Sát Siêu Tham Số: Độ Sâu Cây vs Độ Chính Xác', fontweight='bold')
plt.xlabel('Độ sâu cây tối đa (max_depth)')
plt.ylabel('Độ chính xác (Accuracy)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


## 17. Thí Nghiệm 3: Khảo Sát Biểu Diễn Đặc Trưng (Experiment 3: Representation / Feature Investigation)

**Câu hỏi thực nghiệm:** Việc chuẩn hóa thang đo (`StandardScaler`) và chọn lọc đặc trưng (`Feature Selection`) tác động thế nào đến thuật toán?

In [ ]:
# So sánh Unscaled vs Scaled trên k-NN và Logistic Regression
knn_raw = KNeighborsClassifier(n_neighbors=7).fit(X_train_clean, y_train)
acc_knn_raw = knn_raw.score(X_test_clean, y_test)
acc_knn_scaled = m2.score(X_test, y_test)

lr_raw = LogisticRegression(random_state=42, max_iter=1000).fit(X_train_clean, y_train)
acc_lr_raw = lr_raw.score(X_test_clean, y_test)
acc_lr_scaled = m1.score(X_test, y_test)

print("--- HIỆU QUẢ CỦA BIỂU DIỄN DỮ LIỆU (SCALING VS RAW) ---")
print(f"k-NN: Không chuẩn hóa = {acc_knn_raw:.2%}, Có chuẩn hóa = {acc_knn_scaled:.2%} (Tăng {acc_knn_scaled - acc_knn_raw:+.2%})")
print(f"Logistic Reg: Không chuẩn hóa = {acc_lr_raw:.2%}, Có chuẩn hóa = {acc_lr_scaled:.2%} (Tăng {acc_lr_scaled - acc_lr_raw:+.2%})")


## 18. Mô Hình Cuối Cùng & Đóng Gói (Final Model Selection & Persistence)

Lựa chọn Random Forest làm mô hình cuối cùng, lưu vào file `.pkl` để triển khai vào ứng dụng.

In [ ]:
# Lưu trữ mô hình và bộ tiền xử lý
joblib.dump(m4, 'best_diabetes_model.pkl')
joblib.dump(scaler, 'diabetes_scaler.pkl')
joblib.dump(impute_values, 'diabetes_metadata.pkl')
print("Đã lưu thành công: best_diabetes_model.pkl, diabetes_scaler.pkl, diabetes_metadata.pkl")


## 19. Ứng Dụng Chẩn Đoán (Application Pipeline Implementation)

Hiện thực hóa hàm suy luận tiếp nhận mẫu quan sát thô từ người dùng, chuẩn hóa và dự đoán.

In [ ]:
def predict_diabetes(raw_patient_dict):
    loaded_model = joblib.load('best_diabetes_model.pkl')
    loaded_scaler = joblib.load('diabetes_scaler.pkl')
    loaded_impute = joblib.load('diabetes_metadata.pkl')
    
    patient_df = pd.DataFrame([raw_patient_dict])
    for col, med_val in loaded_impute.items():
        if col in patient_df.columns and patient_df[col].iloc[0] == 0:
            patient_df[col] = med_val
            
    scaled_features = loaded_scaler.transform(patient_df)
    pred_class = loaded_model.predict(scaled_features)[0]
    pred_prob = loaded_model.predict_proba(scaled_features)[0][1]
    
    return {
        'Diagnosis': '⚠️ CẢNH BÁO: NGUY CƠ TIỂU ĐƯỜNG' if pred_class == 1 else '✅ AN TOÀN / NGUY CƠ THẤP',
        'Risk_Probability': f'{pred_prob*100:.1f}%',
        'Class': int(pred_class)
    }

print("Đã khởi tạo hoàn tất hàm suy luận ứng dụng predict_diabetes().")


## 20. Minh Chứng Hệ Thống (System Demonstration - 3 Test Cases)

Thực nghiệm kiểm thử trên 3 ca lâm sàng đại diện.

In [ ]:
test_cases = [
    {'Name': 'Ca 1: Nữ trung niên nguy cơ cao', 'data': {'Pregnancies': 6, 'Glucose': 168, 'BloodPressure': 88, 'SkinThickness': 34, 'Insulin': 230, 'BMI': 34.2, 'DiabetesPedigreeFunction': 0.85, 'Age': 52}},
    {'Name': 'Ca 2: Nữ thanh niên khỏe mạnh', 'data': {'Pregnancies': 1, 'Glucose': 85, 'BloodPressure': 66, 'SkinThickness': 20, 'Insulin': 70, 'BMI': 21.4, 'DiabetesPedigreeFunction': 0.18, 'Age': 24}},
    {'Name': 'Ca 3: Tiền tiểu đường', 'data': {'Pregnancies': 3, 'Glucose': 130, 'BloodPressure': 76, 'SkinThickness': 25, 'Insulin': 110, 'BMI': 27.5, 'DiabetesPedigreeFunction': 0.45, 'Age': 40}}
]

for tc in test_cases:
    res = predict_diabetes(tc['data'])
    print("=== " + tc["Name"] + " ===")
    print("  Kết quả: " + res["Diagnosis"] + " | Xác suất: " + res["Risk_Probability"])


## 21. Phản Ánh & Chiêm Nghiệm (Reflection on Intelligence & Representation)

**7 Câu hỏi bản chất thông minh:**
1. *Hệ thống nhận gì?* Các đo đạc sinh hóa y tế thực tế.
2. *Biểu diễn nội bộ?* Vector $\mathbf{x} \in \mathbb{R}^8$ sau StandardScaler.
3. *Mô hình học gì?* Phân phối xác suất $P(y=1|\mathbf{x})$ và ranh giới phân tách phi tuyến.
4. *Khái quát hóa trên dữ liệu mới?* Nhờ tối ưu hóa hàm mất mát trên phân phối thống kê thay vì ghi nhớ mẫu cụ thể.
5. *Cái gì tạo nên tính thông minh?* Toàn bộ pipeline tích hợp (Biểu diễn + Mô hình + Ứng dụng + Tri thức y khoa Long Châu).

**8 Câu hỏi biểu diễn dữ liệu:**
• Vector đặc trưng gọn nhẹ nhưng làm mất quan hệ cấu trúc không gian và ngữ nghĩa.
• Biểu diễn dạng Đồ thị tri thức (Knowledge Graph) giúp bổ khuyết mối quan hệ nguyên nhân - kết quả giữa Bệnh lý $\leftrightarrow$ Dược phẩm $\leftrightarrow$ Triệu chứng.

In [ ]:
print("Reflection: Trained Model != Complete Intelligent System.")
print("Intelligence emerges from: Representation + Learning + Knowledge Graph + Interaction.")


## 22. Kết Luận & Lộ Trình Phát Triển (Conclusion & Course Roadmap)

Assignment 01 đã hoàn thành viên gạch đầu tiên: $\text{Structured Data} \rightarrow \text{Feature Vectors} \rightarrow \text{Traditional ML} \rightarrow \text{Application}$.

**Lộ trình phát triển môn học (A1 $\rightarrow$ A5):**
• **A1:** Feature vectors + Traditional Machine Learning.
• **A2:** Data preparation + evaluation + improved representations.
• **A3:** Learned representations + Deep Learning (PyTorch).
• **A4:** Tensors + CNN cho ảnh + Đồ thị tri thức có cấu trúc (Knowledge Graphs).
• **A5:** Embeddings + RAG + Hệ thống tương tác đa tác tử (Agentic AI).

In [ ]:
print("ASSIGNMENT 01 DIABETES NOTEBOOK COMPLETED SUCCESSFULLY!")
